## Implement Context Compaction

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    anthropic==0.120.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Create the Anthropic Client

In [ ]:
import anthropic

# Anthropic client through Databricks
client = anthropic.Anthropic(
    api_key="unused",
    base_url=f"{workspace_host}/serving-endpoints/anthropic",
    default_headers={
        "Authorization": f"Bearer {token}"
    }
)

MODEL = "databricks-claude-sonnet-5"

### Define the Initial Context

In [ ]:
initial_prompt = """
You are helping me design an e-commerce platform called Project Falcon.

Remember these requirements throughout our conversation:

- Backend: Python
- Database: PostgreSQL
- Cache: Redis
- Cloud provider: Azure
- Authentication: Microsoft Entra ID
- Maximum infrastructure budget: $8,000/month
- Must support 100,000 concurrent users
- Customer payment information must never be stored directly

The platform needs product catalog, shopping cart, inventory,
checkout, payments, recommendations, and order management services.

Help me design the architecture while respecting these requirements.
"""

messages = [
    {
        "role": "user",
        "content": initial_prompt
    }
]

### Build the Context Compaction Function

In [ ]:
# Low threshold for demonstration
COMPACTION_THRESHOLD = 5000

# Keep the latest 4 messages unchanged
RECENT_MESSAGES_TO_KEEP = 4


def estimate_tokens(messages):
    """
    Rough token estimate for demonstration purposes.
    """
    characters = sum(
        len(message["content"])
        for message in messages
    )

    return characters // 4


def compact_context(messages):

    conversation = "\n\n".join(
        f"{message['role'].upper()}:\n{message['content']}"
        for message in messages
    )

    prompt = f"""
You are performing context compaction for a long-running AI conversation.

Compress the conversation below while preserving everything needed
to continue the task.

PRESERVE:
- Original user requirements
- Important facts and numbers
- Technical and architecture decisions
- Constraints
- Current task state
- Unresolved issues

REMOVE:
- Repetition
- Conversational filler
- Verbose explanations
- Redundant examples

Do not invent information.

Return a concise structured summary.

CONVERSATION:

{conversation}
"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=1500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )

### Create the Conversation Loop

In [ ]:
def chat(user_message):

    global messages

    messages.append({
        "role": "user",
        "content": user_message
    })

    tokens_before = estimate_tokens(messages)

    print(f"\nEstimated context: {tokens_before} tokens")

    # Compact context if threshold is reached

    if tokens_before >= COMPACTION_THRESHOLD:

        print("\nContext threshold reached — compacting...")

        # Keep recent messages unchanged
        old_messages = messages[:-RECENT_MESSAGES_TO_KEEP]
        recent_messages = messages[-RECENT_MESSAGES_TO_KEEP:]

        # Compact older conversation
        summary = compact_context(old_messages)

        # Replace old history with compacted state
        messages = [
            {
                "role": "user",
                "content": f"""
The following is compacted context from our earlier conversation.

Treat it as authoritative project state.

{summary}
"""
            }
        ] + recent_messages

        tokens_after = estimate_tokens(messages)

        print(f"Before compaction : {tokens_before}")
        print(f"After compaction  : {tokens_after}")
        print(f"Tokens removed    : {tokens_before - tokens_after}")

    # Generate Response
    response = client.messages.create(
        model=MODEL,
        max_tokens=1200,
        messages=messages
    )

    answer = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )

    messages.append({
        "role": "assistant",
        "content": answer
    })

    print("\nASSISTANT:\n")
    print(answer)

    return answer

### Simulate a Long Running Conversation

In [ ]:
questions = [
    """
Design the overall API and microservices architecture for Project Falcon.
Explain the responsibilities of each major service and how they communicate.
""",

    """
Now design the product catalog service in detail.
Explain its APIs, database interactions, caching strategy,
scaling approach, and failure handling.
""",

    """
Design the shopping cart service.
Explain how cart state should be managed, how Redis should be used,
how carts expire, and how the service should scale.
""",

    """
Now design inventory management across multiple warehouses.
Discuss inventory reservations, consistency, concurrency,
failure scenarios, and communication with other services.
""",

    """
Design the complete checkout workflow.
Explain every major step from the user's cart through order creation,
inventory reservation, payment processing, and confirmation.
""",

    """
Design the payment architecture.
Explain security, retries, idempotency, payment failures,
and how we should comply with our existing payment-data constraint.
""",

    """
Now design the recommendation system.
Explain the data flow, serving architecture, personalization,
integration with the product catalog, and scaling strategy.
""",

    """
Design observability for the entire platform.
Cover logs, metrics, distributed tracing, alerting,
service health, and troubleshooting production failures.
""",

    """
Design the deployment and scaling architecture.
Explain how each service should scale independently and how
the system should respond to sudden traffic spikes during major sales.
""",

    """
Finally, design the order management architecture.
Explain how orders interact with checkout, inventory, payments,
notifications, and failure recovery.
"""
]


for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 70)
    print(f"TURN {i}")
    print("=" * 70)

    chat(question)

### Test whether the important context survived

In [ ]:
retention_test = """
Let's test your memory of the original Project Falcon requirements.

Without me repeating any requirements, tell me:

1. Which backend language are we using?
2. Which database are we using?
3. Which cache are we using?
4. Which cloud provider are we using?
5. Which authentication system are we using?
6. What is our maximum monthly infrastructure budget?
7. How many concurrent users must we support?
8. Can we directly store customer payment information?

Answer concisely.
"""

chat(retention_test)